# Den 4: Zpracování textových dat – Cvičení: Bag of Words & Klasifikace

## Zadání úkolu:
1. Pomocí knihovny **Pandas** načtěte dataset `imdb_reviews_lemmatized.csv`.
2. Na základě sloupce `review_lemmatized` vytvořte vektory (numerickou reprezentaci textu). Použijte třídu `CountVectorizer` ze Scikit-learn a při inicializaci nastavte parametr `max_features=10000` (vektory budou vytvořeny na základě 10 000 nejčastějších slov v korpusu).
3. Vytvořte DataFrame a rozdělte data na trénovací a testovací sadu (pomocí `train_test_split`).
4. Pomocí modelu **Logistické regrese** (`LogisticRegression(max_iter=1000)`) natrénujte klasifikátor pro predikci pozitivního nebo negativního sentimentu.
5. Natrénujte algoritmus na trénovacích datech.
6. Proveďte predikci na testovacích datech.
7. Vygenerujte `classification_report` a detailně analyzujte výsledky modelu.

### Krok 1: Import knihoven a načtení dat

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Načtení dat
data_path = Path("../data/imdb_reviews_lemmatized.csv")
if not data_path.exists():
    data_path = Path("data/imdb_reviews_lemmatized.csv")

imdb_reviews = pd.read_csv(data_path)
print(f"Načteno recenzí: {len(imdb_reviews)}")
imdb_reviews.head()

### Krok 2: Vektorizace pomocí `CountVectorizer(max_features=10000)`
Převedeme texty na numerické vektory četností termínů (Bag of Words) s omezením na 10 000 nejvýznamnějších slov.

In [ ]:
vectorizer = CountVectorizer(max_features=10000)
X_vectors = vectorizer.fit_transform(imdb_reviews["review_lemmatized"].fillna(""))

print(f"Tvar matice příznaků: {X_vectors.shape}")
print(f"Velikost slovníku: {len(vectorizer.get_feature_names_out()):,} slov")

# Ukázka několika slov ze slovníku
print("Ukázka slovníku:", vectorizer.get_feature_names_out()[:20])

### Krok 3: Příprava dat a rozdělení na Train / Test (80 / 20)
Převedeme cílovou proměnnou sentiment na binární hodnoty (1 = positive, 0 = negative).

In [ ]:
y = imdb_reviews["sentiment"].apply(lambda s: 1 if str(s).lower() == "positive" else 0)

X_train, X_test, y_train, y_test = train_test_split(
    X_vectors, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Trénovací vzorky: {X_train.shape[0]}")
print(f"Testovací vzorky: {X_test.shape[0]}")

### Krok 4 & 5: Trénování modelu Logistické regrese
Vytvoříme instanci `LogisticRegression` s parametrem `max_iter=1000` a natrénujeme model na trénovací sadě.

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
print("Model úspěšně natrénován!")

### Krok 6 & 7: Predikce a vyhodnocení modelu (`classification_report`)

In [ ]:
y_pred = clf.predict(X_test)

print(f"Celková přesnost (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f} %\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative (0)", "Positive (1)"]))

### Krok 8: Interpretace koeficientů – Co model naučil o sentimentu?
Zobrazíme slova s nejvyšší kladnou vahou (silně pozitivní) a nejvyšší zápornou vahou (silně negativní).

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

df_coefs = pd.DataFrame({
    "word": feature_names,
    "weight": coefs
})

print("=== TOP 10 POZITIVNÍCH SLOV ===")
print(df_coefs.sort_values("weight", ascending=False).head(10).to_string(index=False))

print("\n=== TOP 10 NEGATIVNÍCH SLOV ===")
print(df_coefs.sort_values("weight", ascending=True).head(10).to_string(index=False))